# 01b — Detailed EDA (appendix)

The main notebook is `01_market_dynamics_and_spillover_baseline.ipynb` (basic EDA, preprocessing check and the Diebold–Yilmaz baseline). This appendix holds the longer analysis behind the design decisions: stationarity, predictability, feature scaling, distribution shift, edge-threshold choice, lead-lag, event studies and naive baselines. It reads only the pipeline outputs and runs in about a minute; interactive copies of the charts are saved to `reports/figures/*.html`.

In [1]:
import cryptograph  # imports torch first (Windows DLL-order guard)
import json, time, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, torch
from scipy import stats
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from cryptograph.data.config import load_config
from cryptograph.data.graphs.correlation import rolling_corr_mats
from cryptograph.analysis import stats as S, dependence as DEP, event_study as ES, baselines as NB, connectedness as CN

pio.renderers.default = "plotly_mimetype+notebook_connected"
pio.templates.default = "plotly_white"
T0 = time.time()
cfg = load_config(); FIG = cfg.figures_dir; FIG.mkdir(parents=True, exist_ok=True)
pd.set_option("display.width", 200, "display.precision", 3)

def save(fig, name):
    fig.write_html(FIG / f"{name}.html", include_plotlyjs="cdn")
    return fig

SHORT = {"2017 bubble peak": "2017 peak", "2018 crash capitulation": "2018 crash", "COVID Black Thursday": "COVID",
         "China crypto services ban": "China ban", "Terra UST depeg / LUNA collapse": "Terra/LUNA",
         "Celsius withdrawal freeze / Three Arrows": "Celsius/3AC", "FTX collapse": "FTX", "SVB failure / USDC depeg": "SVB/USDC",
         "US spot BTC ETF approval": "BTC ETF", "Yen carry-trade unwind": "Yen unwind", "Oct-10 liquidation cascade": "Oct-10 cascade"}

def add_events(fig, label=True):
    for _, e in events.iterrows():
        fig.add_vline(x=e["date"], line_width=1, line_dash="dash", line_color="firebrick", opacity=0.6)
        if label:
            fig.add_annotation(x=e["date"], y=1, yref="paper", text=SHORT.get(e["name"], e["name"]), showarrow=False,
                               textangle=-90, xanchor="right", yanchor="top", font=dict(color="firebrick", size=10))
    return fig

master = pd.read_parquet(cfg.interim_dir / "master.parquet")
panel = torch.load(cfg.processed_dir / "panel.pt", weights_only=False)
snap = torch.load(cfg.processed_dir / "graph_snapshots.pt", weights_only=False)
events = pd.read_csv(cfg.external_dir / "events.csv", parse_dates=["date"])
audit = pd.read_csv(cfg.reports_dir / "universe_audit.csv")
quality = pd.read_csv(cfg.reports_dir / "data_quality_report.csv").set_index("coin")
xsrc = pd.read_csv(cfg.reports_dir / "cross_source_report.csv").set_index("coin")
counts = pd.read_csv(cfg.reports_dir / "sample_counts.csv")
balance = pd.read_csv(cfg.reports_dir / "label_balance.csv")
splits = json.loads((cfg.processed_dir / "splits.json").read_text())

dates = pd.DatetimeIndex(panel["dates"]); coins = panel["coins"]; FN = panel["feature_names"]; HZ = panel["horizons"]
panelA = [c for c, a in zip(coins, panel["panel_A"]) if a]
panelB = [c for c, b in zip(coins, panel["panel_B"]) if b]
event_coins = [c for c, e in zip(coins, panel["is_event_coin"]) if e]
gB = [coins.index(c) for c in panelB]
day_split = np.array(panel["day_split"])
SM = {s: panel["sample_mask"][s].numpy() for s in ("train", "val", "test")}
A_START = pd.Timestamp(cfg.universe.panel_a_start_date)
close = master["close"].unstack("coin")[coins]
r = np.log(close).diff()
X_raw = panel["X_raw"].numpy(); X_mask = panel["X_mask"].numpy(); active = panel["active"].numpy()
fi = {n: i for i, n in enumerate(FN)}
SPLIT_COLORS = {"train": "#2a6fb0", "val": "#d9822b", "test": "#3a9e5f"}
print(f"{len(dates)} days ({dates[0].date()} .. {dates[-1].date()}), {len(coins)} coins in the master table; "
      f"Panel A = {len(panelA)} coins, Panel B = {len(panelB)} coins, event coins = {event_coins}")

3652 days (2016-09-28 .. 2026-09-27), 29 coins in the master table; Panel A = 20 coins, Panel B = 20 coins, event coins = ['LUNC', 'FTT']


## 1. The dataset

In [2]:
uni = audit[audit["coin"].isin(coins)][["coin", "nonstable_rank", "market_cap_usd", "yahoo_first_valid", "median_volume_usd",
                                         "panel_A", "panel_B", "is_event_coin"]].copy()
uni["market_cap_usd"] = (uni["market_cap_usd"] / 1e9).round(1); uni["median_volume_usd"] = (uni["median_volume_usd"] / 1e6).round(0)
uni.columns = ["coin", "rank today", "market cap $B", "Yahoo data from", "median daily volume $M", "Panel A", "Panel B", "event coin"]
uni.reset_index(drop=True)

,coin,rank today,market cap $B,Yahoo data from,median daily volume $M,Panel A,Panel B,event coin
0,BTC,1,1666.1,2016-09-28,23755.0,True,True,False
1,ETH,2,323.3,2017-11-09,12484.0,True,True,False
2,BNB,3,101.6,2017-11-09,875.0,True,True,False
3,XRP,4,93.0,2017-11-09,1649.0,True,True,False
4,SOL,5,69.6,2020-04-10,1781.0,False,True,False
5,TRX,6,31.7,2017-11-09,545.0,True,True,False
6,ZEC,7,26.4,2017-11-09,141.0,True,True,False
7,DOGE,9,14.5,2017-11-09,460.0,True,True,False
8,LINK,10,10.3,2017-11-09,356.0,True,True,False
9,XMR,11,10.0,2017-11-09,93.0,True,True,False


In [3]:
src = master["close_source"].unstack("coin")[coins]
rows = []
for c in coins:
    s = src[c]
    for label, key in (("Coin Metrics (close price only)", "coinmetrics"), ("Yahoo (full OHLCV)", "yahoo")):
        d = s.index[s == key]
        if len(d):
            rows.append({"coin": c, "source": label, "start": d.min(), "end": (dates[-1] if key == "yahoo" else d.max()) + pd.Timedelta(days=1)})
tl = pd.DataFrame(rows)
order = panelB + [c for c in panelA if c not in panelB] + event_coins
tl["panel"] = tl["coin"].map(lambda c: "A+B" if c in panelA and c in panelB else ("B" if c in panelB else ("A" if c in panelA else "event")))
tl["label"] = tl["coin"] + " (" + tl["panel"] + ")"
fig = px.timeline(tl, x_start="start", x_end="end", y="label", color="source",
                  category_orders={"label": [f"{c} ({tl.loc[tl.coin == c, 'panel'].iloc[0]})" for c in order]},
                  color_discrete_map={"Coin Metrics (close price only)": "#9ecae1", "Yahoo (full OHLCV)": "#2a6fb0"},
                  title="When does each coin have data, and from which source?")
fig.add_vline(x=A_START, line_dash="dash", line_color="firebrick")
fig.add_annotation(x=A_START, y=1.02, yref="paper", text="Panel A starts 2017-11-09", showarrow=False, font=dict(color="firebrick"))
fig.update_layout(height=650, legend=dict(orientation="h", y=-0.08), yaxis_title=None, xaxis_range=[dates[0], dates[-1] + pd.Timedelta(days=30)])
save(fig, "01_data_coverage").show()

*What this shows:* each bar is one coin. Dark blue = full Yahoo data (open/high/low/close/volume); light blue = only a closing price from Coin Metrics. Before Nov 2017 almost no coin has Yahoo data, which is why the balanced Panel A starts on 2017-11-09. Newer coins (SOL, AVAX, NEAR…) enter later; the graph models handle that with an `is_active` mask.

In [4]:
q = quality.loc["ALL"]
issues = pd.DataFrame({"check": ["Closes filled in from Coin Metrics", "Days Yahoo and Coin Metrics disagree >2% (kept, flagged)",
                                 "Volume spikes >20x median (flagged)", "Stale price 3+ days (flagged)", "Moves >40% in a day: confirmed real",
                                 "Moves >40% in a day: no 2nd source (kept)", "Missing days (left empty)", "Bad prices corrected"],
                       "coin-days": [q["cm_backfill"], q["xsrc_disagree"], q["volume_spike"], q["stale_close"], q["large_return_confirmed"],
                                     q["large_return_unverified"], q["missing_day"], q["bad_tick_corrected"]]}).astype({"coin-days": int})
fig = px.bar(issues.sort_values("coin-days"), x="coin-days", y="check", orientation="h", text="coin-days", log_x=True,
             title=f"Data-quality checks over {int(q['active_days']):,} coin-days (log scale)")
fig.update_layout(height=380, yaxis_title=None)
save(fig, "01_quality_checks").show()
rec = pd.read_parquet(cfg.interim_dir / "reconciled.parquet").reset_index()
by_year = (rec[rec["xsrc_flag"]].groupby(rec["date"].dt.year).size() / rec[rec["xsrc_rel_diff"].notna()].groupby(rec["date"].dt.year).size()).fillna(0)
fig = px.bar(x=by_year.index.astype(str), y=by_year.values * 100, labels={"x": "year", "y": "% of days"},
             title="How often do Yahoo and Coin Metrics disagree by more than 2%?", text=[f"{v:.1f}%" for v in by_year.values * 100])
fig.update_layout(height=350)
save(fig, "01_source_disagreement_by_year").show()

*What this shows:* the first chart counts every quality check; nothing was invented and nothing structurally wrong was found (no high<low, no zero prices). The second shows that the two data sources agree almost perfectly after 2021 but disagree on 10–23% of days in 2017–18, so the earliest years are the least reliable.

In [5]:
fig = px.bar(counts, x="dataset", y="samples", color="split", barmode="group", text_auto=",", color_discrete_map=SPLIT_COLORS,
             title="How many training / validation / test samples does each model get?",
             labels={"dataset": "", "samples": "samples"})
fig.update_layout(height=400)
save(fig, "01_sample_counts").show()
print(f"Train covers {int(SM['train'].sum())} days. LSTM windows overlap in 29 of 30 days, so the effective number of "
      f"independent observations is ~{int(SM['train'].sum())} days x 20 coins, small by deep-learning standards.")

Train covers 2043 days. LSTM windows overlap in 29 of 30 days, so the effective number of independent observations is ~2043 days x 20 coins, small by deep-learning standards.


*What this shows:* LSTM = one 30-day window per coin per day; graph snapshots = one graph per day; TGN events = one event per shock from a shocking coin to every other coin. Samples near split boundaries and inside the 30-day embargo are removed, so no information leaks from test into training.

**Findings.** The master table holds 29 coins × 3,652 days (90,643 active coin-days). All 69 missing days belong to one AVAX gap, which is left inactive; no day needed forward-filling. There are no high<low, open/close-outside-range or zero-volume rows. Of 156 daily moves with |log return| > 0.4, 123 are confirmed by Coin Metrics or a catalogued event, 33 (coins without a second source) are kept unverified, and 28 bad ticks were corrected. Yahoo and Coin Metrics agree at lag 0 for all 21 overlapping coins (median |difference| 0.17%). The >2% disagreement rate is concentrated in 2017 (23%) and 2018 (11%) and falls to about 0.2% from 2022, so early-period prices are the least reliable. **Decision:** the data is small by DL standards. There are 2,043 train days × 20 nodes, and the 28,880 LSTM windows overlap in 29 of 30 days, so the effective sample size is closer to 2,000 than 29,000. Models must be small (roughly 10⁴–10⁵ parameters), with dropout or weight decay, early stopping on validation, and results averaged over several seeds.

## 2. Prices and market crises  *(original chart 1, now with verified event dates)*

In [6]:
selected = ["BTC", "ETH", "SOL", "BNB", "DOGE", "ADA", "LINK", "AVAX"]
growth = close[selected].apply(lambda s: s / s.dropna().iloc[0])
fig = go.Figure()
for c in selected:
    fig.add_trace(go.Scatter(x=growth.index, y=growth[c], mode="lines", name=c))
add_events(fig)
fig.update_layout(title="Growth of $1 invested at each coin's first available price (log scale), with major shocks",
                  yaxis_type="log", yaxis_title="growth factor", height=600, hovermode="x unified")
save(fig, "02_price_growth_events").show()

*What this shows:* prices move over several orders of magnitude and every big shock (dashed lines, dates verified in `data/external/events.csv`) hits all coins at once. Prices are non-stationary, so the models use daily log returns and volatility instead of raw prices.

In [7]:
mcap = master["market_cap"].unstack("coin")[coins]
cap_coins = mcap.notna().sum()[lambda s: s > 1500].index.tolist()
dom = (mcap[cap_coins]["BTC"] / mcap[cap_coins].sum(axis=1, min_count=5)).dropna() * 100
fig = px.line(x=dom.index, y=dom.values, labels={"x": "", "y": "BTC share of market cap (%)"},
              title=f"Bitcoin dominance among the {len(cap_coins)} coins with Coin Metrics market caps")
fig.update_layout(height=350)
save(fig, "02_btc_dominance").show()

*What this shows:* BTC's share fell to ~30% at the 2018 altcoin peak and is ~76% today, so the market's composition, and therefore the graph, changes over time.

**Findings.** BTC's share of market cap among the cap-covered master coins ranges from 30% (2018-01-03, altcoin peak) to 76% today, so the cross-section's composition changes a lot over the window. Survivorship is strong. Of the 10 largest coins in late 2016, only BTC, ETH and XRP keep their rank. LTC falls from 5 to 11, DASH from 8 to 14, and DGB from 10 to 17, while today's Panel B includes coins that did not exist in 2016 (SOL, AVAX, NEAR). **Decision:** Panel B (selected by today's ranking) is survivorship-biased toward winners. Keep the event coins LUNC and FTT in the data, report results per split or era rather than pooled, and never read performance on the early train years as representative.

## 3. What can be predicted? Returns vs volatility

In [8]:
x = r["BTC"].dropna() * 100
grid = np.linspace(x.min(), x.max(), 400)
fig = go.Figure()
fig.add_trace(go.Histogram(x=x, histnorm="probability density", nbinsx=150, name="BTC daily returns", marker_color="#2a6fb0"))
fig.add_trace(go.Scatter(x=grid, y=stats.norm.pdf(grid, x.mean(), x.std()), name="normal distribution (same mean/std)", line=dict(color="firebrick")))
fig.update_layout(title=f"BTC daily returns have fat tails (excess kurtosis {stats.kurtosis(x):.0f}; normal = 0)",
                  xaxis_title="daily return (%)", yaxis_title="density (log scale)", yaxis_type="log", height=420)
save(fig, "03_btc_return_fat_tails").show()
dist = S.return_distribution_table(r[panelB])
print(f"Across Panel B: median excess kurtosis {dist['excess_kurtosis'].median():.1f}; normality rejected for {(dist['jarque_bera_p'] < 0.01).sum()}/{len(dist)} coins")

Across Panel B: median excess kurtosis 18.2; normality rejected for 20/20 coins


*What this shows:* on a log scale the red normal curve drops to ~0 beyond ±10%, but BTC has many days beyond ±15%. Extreme days are far more common than a normal model assumes, hence robust (Huber) losses and clipping of model inputs.

In [9]:
ra, sa, lb = S.acf_summary(r[panelB], nlags=10)
acf = pd.DataFrame({"lag (days)": list(range(1, 11)) * 2,
                    "autocorrelation": list(ra.mean(axis=1)) + list(sa.mean(axis=1)),
                    "series": ["returns (direction)"] * 10 + ["squared returns (volatility)"] * 10})
fig = px.bar(acf, x="lag (days)", y="autocorrelation", color="series", barmode="group",
             color_discrete_map={"returns (direction)": "#2a6fb0", "squared returns (volatility)": "#d9822b"},
             title="Does today predict tomorrow? Average over the 20 Panel B coins")
fig.add_hline(y=0, line_color="black", line_width=1)
fig.update_layout(height=400)
save(fig, "03_acf_returns_vs_volatility").show()
adf = S.adf_table(close[panelB], r[panelB])
print(f"ADF test: prices non-stationary for {(adf['p_logprice'] > 0.05).sum()}/{len(adf)} coins, returns stationary for {(adf['p_return'] < 0.01).sum()}/{len(adf)}")

ADF test: prices non-stationary for 15/20 coins, returns stationary for 20/20


*What this shows:* blue bars (return direction) are ≈0 at every lag, so yesterday's return says almost nothing about today's direction. Orange bars (volatility) are clearly positive: calm days follow calm days and turbulent days follow turbulent days. Volatility and shocks are learnable targets; return direction mostly is not.

**Findings.** Returns are extremely heavy-tailed: median excess kurtosis across the 20 Panel B coins is 18, and Jarque-Bera rejects normality for 20/20 coins. Log prices have a unit root (not rejected for 15/20 coins), while returns are stationary (20/20). Returns are almost unpredictable: mean lag-1 ACF is −0.024 and mean |ACF| over 20 lags is 0.026, and P(tomorrow's sign = today's) = 0.469. Volatility clusters strongly: the lag-1 ACF of squared returns is +0.18, Ljung-Box is significant for 20/20 coins, and Spearman(|r_t|, |r_t+1|) = 0.22. **Decision:** models consume returns and volatility features, not prices. Magnitude targets (realized volatility, shock occurrence) are realistically learnable; return direction is close to a coin flip and serves as a sanity target only.

## 4. Targets and class balance

In [10]:
sh = balance[balance["label"].str.startswith("shock_next")].copy()
fig = px.bar(sh, x="split", y=sh["rate"] * 100, color="split", color_discrete_map=SPLIT_COLORS, text=[f"{v:.1%}" for v in sh["rate"]],
             title=f"Next-day shock label (|z| > {panel['shock_threshold']}): only ~4% positives", labels={"y": "% positive"})
fig.update_layout(height=350, showlegend=False)
save(fig, "04_shock_label_rate").show()
ct = balance[balance["label"].str.startswith("contagion")].copy()
ct["k"] = ct["label"].str.extract(r"k(\d)")[0].astype(int)
ct = ct[ct["split"] == "train"]
long = pd.concat([pd.DataFrame({"k (days)": ct["k"], "rate %": ct["rate"] * 100, "": "any coin pair"}),
                  pd.DataFrame({"k (days)": ct["k"], "rate %": ct["rate_given_source_shock"] * 100, "": "given coin i just shocked"})])
fig = px.bar(long, x="k (days)", y="rate %", color="", barmode="group", text_auto=".1f",
             title="Contagion label (train): P(coin j shocks within k days) — much higher after coin i shocks")
fig.update_layout(height=380)
save(fig, "04_contagion_label_rate").show()

*What this shows:* both labels are highly imbalanced (4% shocks; <1% contagion pairs), so accuracy is meaningless and we use weighted/focal loss and AUPRC. Right chart: after a shock at coin i, the chance that coin j also shocks within 1–3 days jumps from <1% to 6–14%. That is the contagion signal the graph model has to learn.

**Findings.** One-day forward returns have excess kurtosis 28 on train, falling to 8 at 30 days. Forward realized volatility is right-skewed (skew 5.1 at h=1) and roughly log-normal. The next-day shock label (|z| > 2.5 against the trailing 30-day σ) is positive 3.9% / 4.1% / 3.9% of the time on train / val / test. Contagion labels are rare unconditionally (0.2–0.8%) but reach 6–11% (k=1) and 14–19% (k=3) given a source shock. That is 1.5–5× the base rate, so a real signal exists. **Decision:** use Huber loss for scaled returns. Model volatility in log space (MSE or Huber on log RV). Use class-weighted BCE (pos_weight ≈ 24 for shocks) or focal loss for the shock and contagion labels. Report AUPRC next to AUROC, because accuracy is meaningless at a 4% base rate.

## 5. Features: scaling and redundancy

In [11]:
from cryptograph.data.features.scaling import RobustScalerSet
sc = RobustScalerSet.fit(X_raw, day_split == "train", FN, coins)
Xpc = sc.transform(X_raw, "per_coin"); Xgl = sc.transform(X_raw, "global")
show_c = ["BTC", "ETH", "DOGE", "LINK", "HBAR"]
rows = []
for lab, arr in (("raw log volume", X_raw), ("global scaling", Xgl), ("per-coin scaling (chosen)", Xpc)):
    for c in show_c:
        v = arr[:, coins.index(c), fi["log_volume"]]
        v = v[np.isfinite(v)][::7]
        rows.append(pd.DataFrame({"value": v, "coin": c, "version": lab}))
bx = pd.concat(rows)
fig = px.box(bx, x="coin", y="value", color="coin", facet_col="version", points=False,
             title="Log volume per coin: per-coin scaling puts every coin on the same scale")
fig.update_yaxes(matches=None, showticklabels=True); fig.update_layout(height=420, showlegend=False)
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
save(fig, "05_scaling_per_coin").show()

*What this shows:* raw volumes differ by orders of magnitude between coins (left). Global scaling keeps those level differences (middle), while per-coin scaling fitted on training dates only centres every coin (right), so the network learns abnormal volume for each coin instead of a coin's identity.

In [12]:
tr = X_raw[SM["train"]][:, gB].reshape(-1, len(FN))
cor = pd.DataFrame(tr, columns=FN).corr(method="spearman").round(2)
fig = px.imshow(cor, text_auto=".1f", color_continuous_scale="RdBu_r", zmin=-1, zmax=1, aspect="auto",
                title="Correlation between input features (train): the volatility features are near-duplicates")
fig.update_layout(height=560)
save(fig, "05_feature_correlation").show()
miss = {s: 1 - (X_mask[SM[s]][:, gB] & active[SM[s]][:, gB][..., None]).sum(axis=(0, 1)) / active[SM[s]][:, gB].sum() for s in SM}
print("share of missing values (masked):", pd.DataFrame(miss, index=FN).loc[["parkinson_7", "log_volume", "mcap_share"]].round(3).to_dict())

share of missing values (masked): {'train': {'parkinson_7': 0.075, 'log_volume': 0.073, 'mcap_share': 0.236}, 'val': {'parkinson_7': 0.0, 'log_volume': 0.0, 'mcap_share': 0.398}, 'test': {'parkinson_7': 0.0, 'log_volume': 0.0, 'mcap_share': 0.462}}


*What this shows:* dark red squares off the diagonal are redundant pairs (Parkinson vs Garman-Klass 0.99, the rolling volatility windows 0.85+). We can drop some of them without losing information, which matters with this little data.

**Findings.** Measured as between-coin level spread relative to within-coin spread on validation, per-coin robust scaling removes level differences that global scaling keeps. For `mcap_share` the ratio drops from 18.6 to 2.5, and global scaling leaves 10% of values beyond ±5. For `log_volume` it drops from 2.37 to 1.35. Return-like features are already comparable across coins, where global scaling is marginally better. Per-coin scaling is therefore chosen, with a global train fallback for the 9 coins that have fewer than 180 train rows on some feature. Several features are redundant: Parkinson vs Garman-Klass ρ = 0.99, rv_7 vs Parkinson 0.91, adjacent RV windows 0.85, RSI vs MACD 0.85. The masks are essential for `mcap_share` (missing 24% / 40% / 46% by split, because Coin Metrics covers few coins and ends 2026-05-24) and for range and volume features in the close-only 2016–17 period (about 7.5% of train rows). Every single-feature mutual information with the next-day shock is ≤ 0.005 nats. **Decision:** drop Garman-Klass and rv_14 in the first model iteration, treat `mcap_share` as optional (off by default), and expect gains only from combining features and cross-asset context.

## 6. Train, validation and test look different (distribution shift)

In [13]:
rv = pd.DataFrame(X_raw[:, gB, fi["rv_30"]], index=dates, columns=panelB)
xmed = rv.median(axis=1) * np.sqrt(365) * 100
fig = px.line(x=xmed.index, y=xmed.values, labels={"x": "", "y": "annualised volatility (%)"},
              title="Typical coin volatility (median of 20 coins, 30-day) with the train / validation / test periods")
for s, col in (("train", "#2a6fb0"), ("val", "#d9822b"), ("test", "#3a9e5f")):
    b = splits["splits"][s]
    fig.add_vrect(x0=b["start"], x1=b["end"], fillcolor=col, opacity=0.10, line_width=0,
                  annotation_text=s, annotation_position="top left")
fig.update_layout(height=380)
save(fig, "06_volatility_regimes_splits").show()
reg = pd.DataFrame({s: {"mean volatility %": xmed[day_split == s].mean(), "shock rate %":
                        100 * balance.set_index(["label", "split"]).loc[(f"shock_next(|z|>{panel['shock_threshold']})", s), "rate"]}
                    for s in ("train", "val", "test")}).T.round(1)
reg

,mean volatility %,shock rate %
train,106.9,3.9
val,62.6,4.1
test,71.5,3.9


*What this shows:* training (2016–2022) is far more volatile than validation and test. Raw volatility levels shift by ~40%, but the standardised shock rate stays at ~4% in every split, so scale-free inputs and targets (z-scores, ratios, log volatility) generalise better across periods.

**Findings.** Volatility levels shift strongly across splits. The cross-sectional median 30-day RV is 0.056 in train versus 0.033 in val and 0.037 in test, and no val or test day exceeds the train 80th-percentile volatility regime (20% of train days do). The volatility features have the largest KS shifts (0.26–0.41) after `log_days_listed`, which stays shifted even when capped at 730 days because every test coin is older than two years. Standardised quantities are stable: the shock rate is 3.9% / 4.1% / 3.9%. **Decision:** robust per-coin scaling is necessary but does not remove regime shift. Favour scale-free inputs and targets (z-scores, volatility ratios, log RV), keep the chronological split (each split contains a major shock: Terra in train, FTX in val, Oct-2025 in test), and expect val/test error to be lower than train error simply because those periods are calmer.

## 7. Coins move together: why a graph model  *(original charts 2 and 3)*

In [14]:
rA = r.loc[A_START:, panelA].dropna()
corrA = rA.corr()
from scipy.cluster.hierarchy import leaves_list, linkage
ordr = corrA.index[leaves_list(linkage(1 - corrA.to_numpy()[np.triu_indices(len(corrA), 1)], "average"))]
fig = px.imshow(corrA.loc[ordr, ordr], color_continuous_scale="Blues", zmin=0, zmax=1, aspect="equal",
                title=f"Return correlation between the 20 Panel A coins ({A_START.date()} onward), similar coins grouped")
fig.update_layout(height=620, width=700)
save(fig, "07_correlation_heatmap").show()
off = corrA.to_numpy()[~np.eye(len(corrA), dtype=bool)]
print(f"average pairwise correlation {off.mean():.2f} (min {off.min():.2f}, max {off.max():.2f}); average correlation with BTC {corrA['BTC'].drop('BTC').mean():.2f}")

average pairwise correlation 0.57 (min 0.39, max 0.80); average correlation with BTC 0.63


*What this shows:* every square is at least 0.39 and the average is 0.57, so a large part of each coin's movement is shared with others. A per-coin LSTM cannot see this; a graph model can.

In [15]:
avgc = DEP.average_pairwise_corr(r[panelB], window=60, min_obs=40)
fig = px.line(x=avgc.index, y=avgc.values, labels={"x": "", "y": "average pairwise correlation"},
              title="How strongly do the 20 Panel B coins move together? (60-day rolling average)")
add_events(fig); fig.update_layout(height=420)
save(fig, "07_avg_correlation_over_time").show()
crisis = pd.Series(False, index=dates)
for d in events["date"]:
    crisis.loc[d: d + pd.Timedelta(days=30)] = True
print(f"average correlation in the 30 days after an event: {avgc[crisis].mean():.2f} vs other days: {avgc[~crisis].mean():.2f}")

average correlation in the 30 days after an event: 0.61 vs other days: 0.54


*What this shows:* co-movement is not constant: it rose from ~0.3 in 2017 to 0.5–0.7 from 2018 on, and jumps after crises (0.61 vs 0.54 on other days), so the graph must be rebuilt every day from recent data only.

In [16]:
act_b = snap["active"].numpy()
edges = pd.Series([e.shape[1] // 2 for e in snap["undirected_edge_index"]], index=dates)
fig = go.Figure()
fig.add_trace(go.Scatter(x=edges.index, y=edges, mode="lines", name="edges per day", line=dict(color="#9ecae1", width=1)))
fig.add_trace(go.Scatter(x=edges.index, y=edges.rolling(30).mean(), mode="lines", name="30-day average", line=dict(color="#2a6fb0", width=2)))
fig.add_trace(go.Scatter(x=edges.index, y=act_b.sum(axis=1) * (act_b.sum(axis=1) - 1) / 2, mode="lines", name="max possible (all pairs)",
                         line=dict(color="grey", dash="dot")))
add_events(fig)
fig.update_layout(title=f"Daily graph size: coin pairs with 30-day |correlation| ≥ {cfg.graph.threshold} (plus top-{cfg.graph.top_k} per coin)",
                  yaxis_title="edges (coin pairs)", height=450, hovermode="x unified")
save(fig, "07_graph_edges_over_time").show()

*What this shows:* each day the graph connects coin pairs whose recent returns are strongly correlated. Edges grow as more coins are listed (grey line) and spike in panics, which is the dynamic structure the graph and TGN models use.

**Findings.** Co-movement is strong: on Panel A, the mean pairwise return correlation is 0.57 (range 0.39–0.80) and the mean correlation with BTC is 0.63. The 60-day average correlation over Panel B rises in the 30 days after catalogued events (0.61 vs 0.54 otherwise). In contrast, high-volatility regimes show lower correlation (0.43 vs 0.56), because the high-volatility days are mostly 2017–2019, when the market was less integrated. The regime comparison is confounded by market maturation. **Decision:** most of each coin's variance is shared with others, which is information a per-coin LSTM cannot see. The graph structure changes over time, so the graph model should be re-estimated daily from trailing windows only.

## 8. Choosing the graph edges

In [17]:
C, D = rolling_corr_mats(r[panelB].to_numpy(), cfg.graph.rolling_window, cfg.graph.min_obs)
iu = ~np.eye(len(panelB), dtype=bool)
rows = []
for thr in (0.3, 0.4, 0.5, 0.6, 0.7):
    for s in ("train", "val", "test"):
        rows.append({"threshold": thr, "split": s, "density %": 100 * np.nanmean(np.abs(C[day_split == s][:, iu]) >= thr)})
dens = pd.DataFrame(rows)
fig = px.line(dens, x="threshold", y="density %", color="split", markers=True, color_discrete_map=SPLIT_COLORS,
              title="Share of coin pairs connected, by correlation threshold")
fig.add_vline(x=cfg.graph.threshold, line_dash="dash", line_color="firebrick")
fig.add_annotation(x=cfg.graph.threshold, y=95, text=f"chosen: {cfg.graph.threshold}", showarrow=False, xanchor="left", font=dict(color="firebrick"))
fig.update_layout(height=380, yaxis_range=[0, 100])
save(fig, "08_threshold_sensitivity").show()

*What this shows:* at a low threshold (0.3–0.4) almost every pair is connected in val/test, so the graph carries no structure. At 0.6 roughly half the pairs remain, enough to keep structure without being a complete graph. We chose 0.6.

In [18]:
se = 1 / np.sqrt(cfg.graph.rolling_window - 3)
thr = 0.3
null = 100 * (1 - stats.norm.cdf(np.arctanh(thr) / se))
obs = {s: 100 * np.nanmean(D[day_split == s][:, iu] > thr) for s in ("train", "val", "test")}
bars = pd.DataFrame({"": ["train", "val", "test", "expected by pure chance"], "% of pairs": [*obs.values(), null]})
fig = px.bar(bars, x="", y="% of pairs", text_auto=".1f", color="", color_discrete_map={**SPLIT_COLORS, "expected by pure chance": "grey"},
             title=f"Directed 'coin i yesterday → coin j today' edges (30-day lag correlation > {thr}) are no more common than chance")
fig.update_layout(height=380, showlegend=False)
save(fig, "08_directed_edges_vs_chance").show()
others = r[[c for c in panelB if c != "BTC"]]
ccf = DEP.cross_correlation(r["BTC"], others, lags=range(-5, 6)).mean(axis=1)
fig = px.bar(x=ccf.index, y=ccf.values, labels={"x": "lag in days (positive = BTC moves first)", "y": "average correlation"},
             title="Does BTC lead the other coins? Only the same-day correlation is large")
fig.update_layout(height=350)
save(fig, "08_btc_lead_lag").show()
Pg, Pbh = DEP.granger_matrix(r.loc[A_START:, panelB], lag=1)
print(f"Granger causality over the full sample (BH-corrected): {int((Pbh < 0.05).sum().sum())} of 380 directed coin pairs significant, but effects are tiny")

Granger causality over the full sample (BH-corrected): 70 of 380 directed coin pairs significant, but effects are tiny


*What this shows:* left: the rule 'connect i→j if yesterday's i predicts today's j' finds fewer edges than pure noise would. Right: BTC's correlation with other coins is ~0.55 on the same day and ≈0 one day later. At daily frequency direction cannot be estimated from short windows, so the TGN takes direction from time order instead: a coin that shocks first sends events to all others (decision agreed on 2026-09-28).

**Findings.** Lead-lag at daily frequency is weak. BTC's cross-correlation with the other Panel B coins is 0.55 at lag 0 but −0.03 at ±1 day and averages 0.02 in magnitude over lags 1–5. Rolling 30-day lag-1 correlations centre on −0.04 with a spread (0.187) matching the null (0.192). Over the full sample, bivariate Granger tests with BH-FDR find 70 of 380 directed pairs significant, 58 of them one-directional. Transmitters include BCH, XMR, XRP and ETH; receivers include DOGE, HBAR and BNB. These effects are tiny and only detectable with about 3,000 observations. **Decision:** direction is real but cannot be estimated from short windows. The TGN therefore gets its direction from time ordering (shock-triggered events: a coin that shocks at t emits events to all active coins) rather than from lag-correlation thresholds. The lag-correlation value is kept as an edge feature.

**Findings.** At threshold 0.4 the undirected graphs were about 75% dense in val and test, close to complete. At the chosen 0.6 (plus top-2 per node), mean density is 59% / 58% / 54% by split with almost no isolated nodes, and without top-k it would be 30% / 55% / 50%, still well above the n = 30 null. The directed lag-1 rule is at or below the null rate at 30-, 90- and 180-day windows (observed/null ratio 0.3–1.3 at τ ≤ 0.2). It exceeds the null only for 365-day windows at τ = 0.2, where it yields about 1 edge per day. **Decision:** undirected threshold 0.6, and a shock-triggered TGN stream with 16,266 / 8,241 / 14,877 events for train / val / test. Events occur on 27% of days, with a median of 19 events on those days and up to 380 on 2026-02-05, when all 20 coins shocked. Lag-1 directed snapshot edges stay available as an option.

## 9. Event studies: how long do shocks last?

In [19]:
rv7 = pd.DataFrame(X_raw[:, :, fi["rv_7"]], index=dates, columns=coins)
fig = go.Figure(); buttons = []; per = []
for k, (_, e) in enumerate(events.iterrows()):
    origin = e["origin_coin"] if isinstance(e["origin_coin"], str) else None
    cols_e = list(dict.fromkeys(panelB + ([origin] if origin else [])))
    cum = (np.exp(ES.event_window(r[cols_e], e["date"], pre=10, post=30)) - 1) * 100  # simple % price change vs day -1
    others_c = [c for c in cols_e if c not in ("BTC", origin)]
    series = {"BTC": cum["BTC"], "median of other coins": cum[others_c].median(axis=1)}
    if origin and origin != "BTC":
        series[f"origin coin: {origin}"] = cum[origin]
    for name, s in series.items():
        fig.add_trace(go.Scatter(x=s.index, y=s.values, name=name, visible=(k == 4),
                                 line=dict(color={"BTC": "black", "median of other coins": "#2a6fb0"}.get(name, "firebrick"), width=2)))
    pr = ES.persistence(r[cols_e], rv7[cols_e], e["date"]); pr = pr[pr.index != origin]
    per.append({"event": SHORT[e["name"]], "date": e["date"].date(), "type": e["event_type"],
                "volatility after / before": pr["vol_ratio"].median(), "days until volatility normal": pr["days_to_normal"].median(),
                "median 30-day price change %": 100 * (np.exp(pr["cum_ret_0_30"].median()) - 1)})
    buttons.append((SHORT[e["name"]], len(series)))
vis_all, start = [], 0
menu = []
total = sum(n for _, n in buttons)
for name, n in buttons:
    vis = [False] * total; vis[start: start + n] = [True] * n; start += n
    menu.append(dict(label=name, method="update", args=[{"visible": vis}, {"title": f"Cumulative return around: {name}"}]))
fig.update_layout(updatemenus=[dict(buttons=menu, direction="down", x=1.0, xanchor="right", y=1.15, active=4)],
                  title="Cumulative return around: Terra/LUNA", xaxis_title="days relative to the event (0 = event day)",
                  yaxis_title="price change since the day before (%)", height=450)
fig.add_vline(x=0, line_dash="dash", line_color="grey")
save(fig, "09_event_explorer").show()

*What this shows:* use the dropdown to pick an event. Black = BTC, blue = the typical other coin, red = the coin where the crisis started (e.g. LUNC for Terra, FTT for FTX). The origin coin collapses far more than the rest, and the others still fall together, which is the contagion we want to model.

In [20]:
per = pd.DataFrame(per)
fig = px.bar(per, x="event", y="volatility after / before", color="type", text_auto=".1f", hover_data=["date", "days until volatility normal"],
             title="How much did volatility rise in the 30 days after each event? (median across coins; 1 = no change)")
fig.add_hline(y=1, line_dash="dash", line_color="grey")
fig.update_layout(height=420, xaxis_title=None)
save(fig, "09_event_volatility_ratio").show()
per.round(2)

,event,date,type,volatility after / before,days until volatility normal,median 30-day price change %
0,2017 peak,2017-12-17,market_structure,1.12,1.0,88.95
1,2018 crash,2018-02-05,market_structure,0.71,1.0,10.88
2,COVID,2020-03-12,macro,1.99,14.0,-12.72
3,China ban,2021-05-19,regulatory,1.42,11.5,-36.94
4,Terra/LUNA,2022-05-09,single_asset_collapse,1.61,11.0,-26.46
5,Celsius/3AC,2022-06-13,exchange_or_lender_failure,1.14,1.0,-14.03
6,FTX,2022-11-08,exchange_or_lender_failure,2.01,9.0,-17.93
7,SVB/USDC,2023-03-10,exchange_or_lender_failure,1.09,1.0,15.83
8,BTC ETF,2024-01-10,regulatory,0.78,1.0,3.20
9,Yen unwind,2024-08-05,macro,1.18,6.0,-1.42


*What this shows:* COVID, FTX, Terra and the Oct-2025 liquidation cascade roughly doubled volatility, and it took ~1–2 weeks to calm down. Regulatory news (BTC ETF) barely moved it. This previews the 'persistence by shock type' objective (only 11 events, so descriptive, not a statistical test).

**Findings.** Across the 11 events, 30-day volatility of the receiving coins rises by a median factor of 1.1–2.0. The strongest effects are FTX (2.0), COVID (2.0), Oct-2025 (1.7) and Terra (1.6), while the 2018 capitulation and the ETF approval see volatility fall. Median time back inside the pre-event 7-day-volatility band is 9–14 days for COVID, China, Terra and FTX, versus about 1 day for SVB, Celsius and the ETF approval. The origin coin collapses far more than the receivers: over the 30 days LUNC lost ≈100% and FTT ≈94%, while the median other coin lost ≈26% (Terra) and ≈18% (FTX). By type, the single-asset collapse (Terra) is the most persistent (11 days, volatility ratio 1.6), then macro shocks (7.5 days, 1.36), with regulatory and lender failures shortest. **Caveat:** the pre-event band is wide when the run-up was already turbulent, so 'days to normal' understates persistence for events like the 2017 peak. **Decision:** persistence clearly differs by shock type, which supports the persistence-by-type objective, but with only 11 events any comparison across types is descriptive, not a test.

## 10. The bar to beat: naive baselines

In [21]:
lc = np.log(close.to_numpy())[:, gB]; rr = r.to_numpy()[:, gB]
yr_all = panel["y_ret"].numpy()[:, gB]; yv_all = panel["y_rv"].numpy()[:, gB]; ymask = panel["y_mask"].numpy()[:, gB]
last_ret = np.full_like(yr_all, np.nan); last_rv = np.full_like(yv_all, np.nan)
sq = pd.DataFrame(rr ** 2)
for k, h in enumerate(HZ):
    last_ret[h:, :, k] = lc[h:] - lc[:-h]
    last_rv[:, :, k] = np.sqrt(sq.rolling(h, min_periods=h).mean().to_numpy())
def train_mean(y):
    m = np.nanmean(np.where(ymask[SM["train"]], y[SM["train"]], np.nan), axis=0)
    return np.where(np.isfinite(m), m, np.nanmean(m, axis=0, keepdims=True))
ysh = panel["y_shock"].numpy()[:, gB]; yshm = panel["y_shock_mask"].numpy()[:, gB]
st = panel["shock_today"].numpy()[:, gB]; stm = panel["shock_today_mask"].numpy()[:, gB]
zabs = np.abs(panel["shock_z"].numpy()[:, gB])
rows = []
for s in ("val", "test"):
    m = SM[s]
    rows += NB.naive_regression(yr_all[m], ymask[m], last_ret[m], train_mean(yr_all), HZ, "fwd_log_return", s)
    rows += NB.naive_regression(yv_all[m], ymask[m], last_rv[m], train_mean(yv_all), HZ, "fwd_realized_vol", s)
    mm = yshm[m] & stm[m]
    rows.append(NB.shock_persistence_rule(st[m], ysh[m], mm, s))
    rows.append(NB.shock_score_rule(zabs[m], ysh[m], mm, s, "abs_z_today_score"))
nb = pd.DataFrame(rows); nb.to_csv(cfg.reports_dir / "naive_baselines.csv", index=False)
v7 = nb[(nb["target"] == "fwd_realized_vol") & (nb["horizon"] == 7)]
names = {"zero": "predict 0", "persistence": "last week's volatility", "historical_mean": "training average"}
fig = px.bar(v7.assign(model=v7["model"].map(names)), x="model", y="MSE", color="split", barmode="group", text_auto=".4f",
             color_discrete_map=SPLIT_COLORS, title="Predicting next-week volatility: simple baselines (lower error is better)")
fig.update_layout(height=380, xaxis_title=None)
save(fig, "10_baseline_volatility").show()
shk = nb[nb["target"] == "shock_next"].copy()
shk["model"] = shk["model"].map({"shock_today_rule": "'shock tomorrow if shock today'", "abs_z_today_score": "size of today's move |z|"})
fig = px.bar(shk, x="model", y="AUROC", color="split", barmode="group", text_auto=".2f", color_discrete_map=SPLIT_COLORS,
             title="Predicting tomorrow's shock: simple rules (AUROC, 0.5 = random guessing)")
fig.add_hline(y=0.5, line_dash="dash", line_color="grey"); fig.update_layout(height=380, yaxis_range=[0.4, 0.7], xaxis_title=None)
save(fig, "10_baseline_shock").show()

*What this shows:* a deep-learning model is only useful if it beats these. For volatility, 'last week's volatility' is already a strong forecast; for shocks, simple rules reach AUROC ≈0.55–0.62. For return direction the best naive forecast is simply 0 (full table in `reports/naive_baselines.csv`).

**Findings.** For returns, the zero forecast is best at every horizon (val MSE 0.0015 at h=1 vs 0.0031 for persistence), and the train mean is no better. For realized volatility, persistence (trailing RV) beats both the train mean and zero from h=3 onward (test MSE at h=7: 0.0007 vs 0.0010 vs 0.0018), consistent with volatility clustering. For next-day shocks, the 'shock today' rule reaches AUROC 0.55 / AUPRC 0.056 on val (base rate 0.041), and the |z_today| score reaches AUROC 0.61–0.62 / AUPRC 0.076–0.087. **Decision:** these numbers, saved in `reports/naive_baselines.csv`, are the bar. A DL model that does not beat zero on returns, persistence on RV, and AUROC 0.62 on shocks adds nothing.

## 11. Statistical spillover baseline: Diebold–Yilmaz  *(original charts 4 and 5, corrected)*

In [22]:
from statsmodels.tsa.api import VAR
sel = VAR(rA).select_order(maxlags=5)
ic = {k: int(v) for k, v in sel.selected_orders.items()}
dy = CN.dy_connectedness(rA, H=10, lags=ic["bic"] if ic["bic"] > 0 else 1)
summ = dy["summary"].sort_values("NET", ascending=False)
print(f"VAR on Panel A, {len(rA)} days, lag order {dy['lag_order']} (BIC chose {ic['bic']}, AIC {ic['aic']}), 10-day horizon, generalized FEVD (order-invariant)")
print(f"TOTAL SPILLOVER INDEX = {dy['total']:.1f}% of each coin's forecast-error variance comes from the other coins")
net_df = pd.DataFrame({"coin": summ.index, "NET": summ["NET"].values,
                       "role": np.where(summ["NET"] > 0, "net transmitter", "net receiver")})
fig = px.bar(net_df, x="coin", y="NET", color="role", color_discrete_map={"net transmitter": "#3a9e5f", "net receiver": "#c8423e"},
             title=f"Who spreads shocks and who absorbs them? NET spillover = TO − FROM (total spillover {dy['total']:.0f}%)",
             labels={"NET": "NET spillover (percentage points)"})
fig.update_layout(height=420)
save(fig, "11_dy_net_ranking").show()
summ.round(1).T

VAR on Panel A, 3244 days, lag order 1 (BIC chose 0, AIC 5), 10-day horizon, generalized FEVD (order-invariant)
TOTAL SPILLOVER INDEX = 86.0% of each coin's forecast-error variance comes from the other coins


C:\Users\HARDIK\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
C:\Users\HARDIK\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)


,ETH,LTC,NEO,BTC,ETC,BCH,ADA,DASH,XLM,ZEC,XMR,XRP,LINK,DGB,BNB,XTZ,LSK,DCR,TRX,DOGE
TO,117.6,107.1,106.5,104.5,102.3,93.9,92.9,88.8,88.7,84.0,83.2,81.9,81.2,79.7,78.4,75.7,68.7,68.6,61.1,54.9
FROM,89.5,88.7,88.5,88.4,88.2,87.3,87.2,86.7,86.7,86.1,85.8,85.8,85.7,85.4,85.4,84.8,83.6,83.6,82.0,80.6
NET,28.1,18.4,18.0,16.1,14.1,6.6,5.7,2.1,2.0,-2.2,-2.6,-3.8,-4.5,-5.7,-7.0,-9.1,-14.8,-15.0,-20.9,-25.7


*What this shows:* green coins send more shocks than they receive (ETH, LTC, NEO, BTC); red coins mostly absorb them (DOGE, TRX). This ranking is saved as the ground truth for validating the TempME explainer. The original notebook's values (BTC +731) came from the order-dependent Cholesky method; the generalized method used here gives the same answer for any column order.

In [23]:
rolling = CN.rolling_total_spillover(rA, window=200, lags=1, H=10, step=5)
fig = px.line(x=rolling.index, y=rolling.values, labels={"x": "", "y": "total spillover (%)"},
              title="Total spillover over time (200-day rolling window)")
add_events(fig); fig.update_layout(height=420)
save(fig, "11_dy_rolling_spillover").show()
out = summ.copy(); out["net_rank"] = np.arange(1, len(out) + 1)
out["total_spillover_index"] = dy["total"]; out["var_lags"] = dy["lag_order"]; out["horizon"] = 10
out["window_start"] = rA.index[0].date(); out["window_end"] = rA.index[-1].date(); out["method"] = "DY2012 generalized FEVD (Pesaran-Shin)"
out.index.name = "coin"; out.to_csv(cfg.path(cfg.paths.processed_dir) / "statistical_connectedness.csv")
rolling.to_csv(cfg.reports_dir / "dy_rolling_total_spillover.csv")
res = VAR(rA).fit(dy["lag_order"])
def chol_net(res, H=10):
    dec = res.fevd(H).decomp[:, H - 1, :]; th = 100 * dec / dec.sum(1, keepdims=True)
    return CN.spillover_table(th, list(res.names))["summary"]["NET"]
n1, n2 = chol_net(res), chol_net(VAR(rA[rA.columns[::-1]]).fit(dy["lag_order"]))
print(f"Why not the old method: reversing the column order changes the Cholesky NET ranking (Spearman {stats.spearmanr(n1, n2.reindex(n1.index))[0]:.2f}); "
      "the generalized FEVD is unchanged (unit-tested).")

C:\Users\HARDIK\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
C:\Users\HARDIK\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
C:\Users\HARDIK\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
C:\Users\HARDIK\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
C:\Users\HARDIK\AppData\Local\Programs\Python\Python312\

C:\Users\HARDIK\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
C:\Users\HARDIK\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
C:\Users\HARDIK\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
C:\Users\HARDIK\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
C:\Users\HARDIK\AppData\Local\Programs\Python\Python312\

C:\Users\HARDIK\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
C:\Users\HARDIK\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
C:\Users\HARDIK\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
C:\Users\HARDIK\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
C:\Users\HARDIK\AppData\Local\Programs\Python\Python312\

C:\Users\HARDIK\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
C:\Users\HARDIK\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
C:\Users\HARDIK\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
C:\Users\HARDIK\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
C:\Users\HARDIK\AppData\Local\Programs\Python\Python312\

C:\Users\HARDIK\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
C:\Users\HARDIK\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
C:\Users\HARDIK\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
C:\Users\HARDIK\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
C:\Users\HARDIK\AppData\Local\Programs\Python\Python312\

...

C:\Users\HARDIK\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
C:\Users\HARDIK\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
C:\Users\HARDIK\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
C:\Users\HARDIK\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
C:\Users\HARDIK\AppData\Local\Programs\Python\Python312\

C:\Users\HARDIK\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
C:\Users\HARDIK\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
C:\Users\HARDIK\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
C:\Users\HARDIK\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
C:\Users\HARDIK\AppData\Local\Programs\Python\Python312\

C:\Users\HARDIK\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
C:\Users\HARDIK\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
C:\Users\HARDIK\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
C:\Users\HARDIK\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
C:\Users\HARDIK\AppData\Local\Programs\Python\Python312\

Why not the old method: reversing the column order changes the Cholesky NET ranking (Spearman -0.49); the generalized FEVD is unchanged (unit-tested).


C:\Users\HARDIK\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
C:\Users\HARDIK\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)


*What this shows:* spillover stays between ~78% and ~92%: crypto is almost always highly interconnected, peaking around the 2020 crash and easing recently.

**Findings.** On Panel A (20 coins, 3,244 days) BIC and HQIC select 0 lags (AIC and FPE select 5), so a VAR(1) is used, the minimal dynamic model. The generalized-FEVD total spillover index is 86%: 80–90% of each coin's 10-day forecast-error variance comes from the others. The rolling 200-day index stays between 78% and 92%, peaking in mid-2020 and at its low at the end of the sample. NET transmitters are ETH (+28), LTC (+18), NEO (+18), BTC (+16) and ETC (+14); the largest receivers are DOGE (−26), TRX (−21), DCR (−15) and LSK (−15). The Cholesky alternative is not usable: reversing the column order changes its NET ranking to Spearman −0.49, with shifts up to 815 pp, while the generalized FEVD is order-invariant (checked in the unit tests). **Decision:** `data/processed/statistical_connectedness.csv` (NET ranking) is the ground truth the TempME explanations will be validated against.

## 12. Summary for the report

**Key findings**

1. Returns are heavy-tailed (median excess kurtosis 18) and almost unpredictable in sign (P(same sign) = 0.47). Volatility clusters (squared-return ACF 0.18) and shocks are partly predictable (|z| rule AUROC 0.61).
2. Coins are strongly and time-varyingly co-dependent: mean pairwise correlation 0.57, and 86% of forecast-error variance is spillover. Correlations rise after crisis events.
3. Daily lead-lag is weak (|CCF| ≈ 0.02–0.03), and rolling lag-1 correlations are indistinguishable from noise. Full-sample Granger finds sparse direction (70/380 pairs).
4. Volatility regimes shift: val and test are about 40% calmer than train, while standardised shock rates stay at about 4% in every split.

**Modelling decisions**

- **Inputs:** 12 trailing-only features with masks; drop Garman-Klass, rv_14 and `mcap_share` in the first iteration. Per-coin robust scaling fitted on train, clipped at ±5 for inputs only.
- **Targets:** log RV and shock or contagion labels as primary targets; returns as a sanity target. Losses: Huber and log-space MSE for regression; class-weighted BCE or focal loss for labels (4% and 0.2–0.8% positives). Report AUPRC.
- **Graphs:** undirected rolling 30-day correlation with |ρ| ≥ 0.6 plus top-2. The TGN uses shock-triggered directed events, with the lag-1 correlation kept as a message feature.
- **Splits:** chronological 2016-09-28 → 2022-06-30 / 2022-07-01 → 2023-12-31 / 2024-01-01 → 2026-09-27, with a 30-day embargo and boundary-crossing samples dropped.
- **Baselines to beat:** zero (returns), persistence (RV), the |z| score (shocks), and the Diebold-Yilmaz NET ranking (explainer validation).

**Limitations**

- Yahoo history starts on 2017-11-09 for all coins except BTC and LTC. Earlier data is close-only Coin Metrics backfill, where it passed validation.
- Cross-source disagreement was 11–23% of days in 2017–18. Eight coins, including SOL, AVAX and LUNC, have no second price source, and 33 large moves are unverified.
- Exchange-reported volume is noisy (194 spike flags). Market caps cover only some coins and end on 2026-05-24.
- Universe selection by today's ranking causes survivorship bias.
- The sample is small (about 2,000 train days), so every DL result needs seeds, confidence intervals and the naive baselines above.

In [24]:
print(f"notebook runtime: {time.time() - T0:.0f} s")

notebook runtime: 9 s
